# Visualização: sífilis congênita em Porto Alegre

Este notebook cria uma visualização simples sobre desigualdade racial na sífilis congênita em Porto Alegre.

A ideia central é comparar a composição racial das mães em duas bases de 2024:

- SINASC: nascidos vivos em Porto Alegre, usado como contexto/denominador.
- SINAN/SIFCBR: notificações de sífilis congênita em residentes de Porto Alegre, usado como desfecho.

A visualização final pergunta: a participação de mães negras entre os casos notificados de sífilis congênita é maior do que entre os nascidos vivos do município?


## 1. Dependências

No Colab, execute esta célula uma vez. Ela remove instalações antigas de `pysus`/`pandas` que podem ter ficado de tentativas anteriores e instala apenas o leitor de arquivos `.dbc` (`pyreaddbc`) com versões compatíveis das bibliotecas de análise.

Depois que a instalação terminar, o notebook reiniciará o runtime automaticamente no Colab. Após o reinício, continue a partir da próxima seção sem repetir esta instalação. Isso evita erro de incompatibilidade binária como `numpy.dtype size changed`.


In [ ]:
!rm -rf /usr/local/lib/python3.12/dist-packages/~andas*
!pip -q uninstall -y pysus numpy pandas matplotlib seaborn python-dateutil
!pip -q install --no-cache-dir --force-reinstall "numpy==1.26.4" "pandas==2.2.2" "python-dateutil==2.8.2" "matplotlib==3.8.4" "seaborn==0.13.2" "pyreaddbc==2.0.4"

import os
import sys

if "google.colab" in sys.modules:
    print("Instalação concluída. O runtime será reiniciado automaticamente.")
    os.kill(os.getpid(), 9)
else:
    print("Instalação concluída. Reinicie o kernel antes de continuar.")


## 2. Bibliotecas e configurações

Se a importação abaixo falhar com erro de `numpy.dtype size changed`, o runtime ainda está carregando binários antigos. Reinicie o ambiente de execução do Colab e execute novamente a partir desta célula, sem repetir a instalação.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

CODIGO_PORTO_ALEGRE = "431490"


## 3. Caminhos dos dados

A estrutura esperada no repositório é:

- `data/raw/SIFCBR24.dbc`
- `data/raw/sinasc/DNRS2024.dbc`
- `data/raw/cnes/STRS24*.dbc`


In [ ]:
def encontrar_arquivo(*candidatos):
    for candidato in candidatos:
        caminho = Path(candidato)
        if caminho.exists():
            return caminho
    caminhos = ", ".join(str(c) for c in candidatos)
    raise FileNotFoundError(f"Nenhum arquivo encontrado. Caminhos testados: {caminhos}")

ARQUIVO_SINAN = encontrar_arquivo(
    "../data/raw/SIFCBR24.dbc",
    "data/raw/SIFCBR24.dbc",
    "SIFCBR24.dbc",
)

ARQUIVO_SINASC = encontrar_arquivo(
    "../data/raw/sinasc/DNRS2024.dbc",
    "data/raw/sinasc/DNRS2024.dbc",
    "DNRS2024.dbc",
)

print("SINAN:", ARQUIVO_SINAN)
print("SINASC:", ARQUIVO_SINASC)


## 4. Leitura dos arquivos `.dbc`

O pacote `pyreaddbc` converte os arquivos comprimidos `.dbc` do DATASUS para tabelas que podem ser manipuladas com pandas.


In [ ]:
try:
    import pyreaddbc
except ImportError as exc:
    raise ImportError(
        "Não foi possível importar pyreaddbc. Execute novamente a célula de dependências."
    ) from exc

sinan = pyreaddbc.read_dbc(str(ARQUIVO_SINAN), encoding="iso-8859-1")
sinasc = pyreaddbc.read_dbc(str(ARQUIVO_SINASC), encoding="iso-8859-1")

print("SINAN", sinan.shape)
print("SINASC", sinasc.shape)


## 5. Recorte: Porto Alegre

Nos microdados, o código de Porto Alegre aparece como `431490` ou com dígito adicional. Por isso o filtro usa `startswith("431490")`.


In [ ]:
sinan_poa = sinan[
    sinan["ID_MN_RESI"].astype(str).str.strip().str.startswith(CODIGO_PORTO_ALEGRE)
].copy()

sinasc_poa = sinasc[
    sinasc["CODMUNRES"].astype(str).str.strip().str.startswith(CODIGO_PORTO_ALEGRE)
].copy()

print("Casos SINAN Porto Alegre:", len(sinan_poa))
print("Nascidos vivos SINASC Porto Alegre:", len(sinasc_poa))


## 6. Preparação das variáveis

Para dialogar com o anteprojeto, a análise separa mães negras de mães não negras. No padrão do SUS, população negra combina as categorias preta e parda.


In [ ]:
def grupo_raca(valor):
    codigo = str(valor).strip()
    if codigo in {"2", "4"}:
        return "Mães negras"
    if codigo in {"1", "3", "5"}:
        return "Mães não negras"
    return "Ignorado/sem informação"

sinan_poa["grupo_raca"] = sinan_poa["ANT_RACA"].map(grupo_raca)
sinasc_poa["grupo_raca"] = sinasc_poa["RACACORMAE"].map(grupo_raca)

base_comparacao = pd.concat(
    [
        sinasc_poa.assign(base="Nascidos vivos (SINASC)")[["base", "grupo_raca"]],
        sinan_poa.assign(base="Casos de sífilis congênita (SINAN)")[["base", "grupo_raca"]],
    ],
    ignore_index=True,
)

base_comparacao = base_comparacao[
    base_comparacao["grupo_raca"].isin(["Mães negras", "Mães não negras"])
]

resumo_raca = (
    base_comparacao
    .groupby(["base", "grupo_raca"], as_index=False)
    .size()
    .rename(columns={"size": "registros"})
)

resumo_raca["percentual"] = resumo_raca.groupby("base")["registros"].transform(
    lambda valores: valores / valores.sum() * 100
)

resumo_raca


## 7. Indicador auxiliar: pré-natal nos casos notificados

Este painel ajuda a manter o gráfico conectado ao argumento do anteprojeto: sífilis congênita é um evento evitável e relacionado a acesso/qualidade do pré-natal.


In [ ]:
mapa_prenatal = {
    "1": "Realizou pré-natal",
    "2": "Não realizou pré-natal",
    "9": "Ignorado/sem informação",
}

prenatal = sinan_poa.copy()
prenatal["grupo_prenatal"] = prenatal["ANT_PRE_NA"].astype(str).str.strip().map(mapa_prenatal)
prenatal["grupo_prenatal"] = prenatal["grupo_prenatal"].fillna("Ignorado/sem informação")

resumo_prenatal = (
    prenatal
    .groupby("grupo_prenatal", as_index=False)
    .size()
    .rename(columns={"size": "casos"})
    .sort_values("casos", ascending=False)
)

resumo_prenatal["percentual"] = resumo_prenatal["casos"] / resumo_prenatal["casos"].sum() * 100
resumo_prenatal


## 8. Visualização final

A figura combina dois painéis: composição racial comparada entre SINASC e SINAN, e situação de pré-natal entre os casos de sífilis congênita.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.25, 1]})

ordem_bases = ["Nascidos vivos (SINASC)", "Casos de sífilis congênita (SINAN)"]
paleta_raca = {
    "Mães negras": "#B55A30",
    "Mães não negras": "#4C78A8",
}

sns.barplot(
    data=resumo_raca,
    x="percentual",
    y="base",
    hue="grupo_raca",
    order=ordem_bases,
    palette=paleta_raca,
    ax=axes[0],
)

axes[0].set_title("Composição racial: nascidos vivos vs. casos notificados", weight="bold")
axes[0].set_xlabel("Percentual dentro de cada base")
axes[0].set_ylabel("")
axes[0].set_xlim(0, 100)
axes[0].legend(title="Grupo racial", loc="lower right")

for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.1f%%", padding=3, fontsize=9)

plot_prenatal = resumo_prenatal.sort_values("percentual", ascending=True)
cores_prenatal = ["#72B7B2" if "Realizou" in item else "#E45756" for item in plot_prenatal["grupo_prenatal"]]

axes[1].barh(plot_prenatal["grupo_prenatal"], plot_prenatal["percentual"], color=cores_prenatal)
axes[1].set_title("Pré-natal entre casos notificados", weight="bold")
axes[1].set_xlabel("Percentual dos casos")
axes[1].set_ylabel("")
axes[1].set_xlim(0, 100)

for i, row in enumerate(plot_prenatal.itertuples(index=False)):
    axes[1].text(row.percentual + 1, i, f"{row.percentual:.1f}%", va="center", fontsize=9)

for ax in axes:
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", alpha=0.25)
    ax.grid(axis="y", visible=False)

fig.suptitle(
    "Sífilis congênita em Porto Alegre (2024): desigualdade racial e acesso ao pré-natal",
    fontsize=14,
    weight="bold",
    y=1.03,
)

plt.tight_layout()

saida_dir = Path("../outputs/figures")
if not saida_dir.exists():
    saida_dir = Path("outputs/figures")
saida_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(saida_dir / "visualizacao_sifilis_congenita_poars.png", dpi=200, bbox_inches="tight")
plt.show()


## 9. Como usar no relatório

Imagem gerada: `outputs/figures/visualizacao_sifilis_congenita_poars.png`.

Caption sugerido para você adaptar com suas palavras: a figura compara a composição racial das mães de nascidos vivos em Porto Alegre com a composição racial das mães associadas a notificações de sífilis congênita no mesmo município e ano. O painel complementar mostra a distribuição dos casos segundo realização de pré-natal, conectando o desfecho a um marcador de acesso ao cuidado.
